# PolicyLens — Ingestion & Retrieval Walkthrough
Runs the full pipeline end to end and shows sample policy-lookup queries against the index.

Run from the repo root: `jupyter notebook notebooks/walkthrough.ipynb`

In [ ]:
import sys, json
sys.path.append('..')
from src import ingest, extract, clean, metadata, chunk, embed_index, retrieve

## Stage 1–2: Ingest raw sources and extract text

In [ ]:
ingest_results = ingest.run()
extract_results = extract.run()

## Stage 3–4: Clean, build metadata manifest, chunk

In [ ]:
clean_records = clean.run()
metadata.run()
chunks = chunk.run()

## Review the metadata manifest

In [ ]:
import pandas as pd
from config import METADATA_CSV
df = pd.read_csv(METADATA_CSV)
df[['source_id','source_title','topic_category','char_count','is_duplicate','low_confidence','missing']]

## Stage 5: Embed chunks and build the FAISS index

In [ ]:
embed_index.run()

## Stage 6: Retrieval smoke test
Sample policy-lookup questions. Each result shows the matched passage, its source, and a similarity score. Low-similarity hits are flagged rather than presented as confident matches.

In [ ]:
sample_queries = [
    "What documents mention eligibility exceptions?",
    "Where is the escalation or appeal process described?",
    "Which source explains required documents for application review?",
    "What procedure applies when information is incomplete?",
]
for q in sample_queries:
    results = retrieve.search(q, top_k=5)
    retrieve.print_results(q, results)

## Inspect one query as a table

In [ ]:
results = retrieve.search("Where is the escalation or appeal process described?", top_k=5)
pd.DataFrame(results)[['score','low_confidence','source_title','issuing_body','topic_category','source_url']]

## Notes for reviewers
- Every retrieved passage carries its `source_url` and `issuing_body` — no answer is presented without a traceable source.
- `low_confidence` is set on both corpus records (short/thin extracted text) and query results (weak similarity score). Treat these as **needs human review**, not as answers.
- This notebook produces retrieval candidates only. It does not interpret, summarize into policy advice, or make eligibility determinations — that is explicitly out of scope.